In [83]:

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

dataset = fetch_openml("house_prices",as_frame=True)

X = dataset.data
y = dataset.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.info())
print(X_train.isnull().sum().sort_values(ascending=False))
print(X_train.dtypes.value_counts())



<class 'pandas.DataFrame'>
Index: 1168 entries, 254 to 1126
Data columns (total 80 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1168 non-null   int64  
 1   MSSubClass     1168 non-null   int64  
 2   MSZoning       1168 non-null   str    
 3   LotFrontage    951 non-null    float64
 4   LotArea        1168 non-null   int64  
 5   Street         1168 non-null   str    
 6   Alley          74 non-null     str    
 7   LotShape       1168 non-null   str    
 8   LandContour    1168 non-null   str    
 9   Utilities      1168 non-null   str    
 10  LotConfig      1168 non-null   str    
 11  LandSlope      1168 non-null   str    
 12  Neighborhood   1168 non-null   str    
 13  Condition1     1168 non-null   str    
 14  Condition2     1168 non-null   str    
 15  BldgType       1168 non-null   str    
 16  HouseStyle     1168 non-null   str    
 17  OverallQual    1168 non-null   int64  
 18  OverallCond    1168 no

In [84]:
numerical_features = X_train.select_dtypes(exclude = ['object']).columns
print(numerical_features)

categorical_features = X_train.select_dtypes(include=['object']).columns
print(categorical_features)

Index(['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd',
       'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF',
       'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea',
       'MiscVal', 'MoSold', 'YrSold'],
      dtype='str')
Index(['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heating', 'HeatingQC', 'Cen

/var/folders/xs/j2zflpbs65bg9wmkm8rmvf_h0000gn/T/ipykernel_24262/1571117867.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(include=['object']).columns


In [85]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy = 'median')),
                                        ('scaler',StandardScaler())])

categorical_transformer = Pipeline(steps = [('imputer',SimpleImputer(strategy='most_frequent')),
('encoder',OneHotEncoder(handle_unknown='ignore'))])


column_transformer = ColumnTransformer([('numerical_transformer',numeric_transformer, numerical_features),('categorical_transformer',categorical_transformer,categorical_features)])

In [86]:
from sklearn.ensemble import RandomForestRegressor

regression_pipeline = Pipeline([('preprocessor',column_transformer),('model',RandomForestRegressor(n_estimators=100,max_depth=20,random_state=42))])
regression_pipeline.fit(X_train,y_train)

y_pred_train = regression_pipeline.predict(X_train)
y_pred_test = regression_pipeline.predict(X_test)

print(y_train.shape)
print(y_pred_train.shape)

(1168,)
(1168,)


In [87]:
from sklearn.metrics import mean_squared_error,r2_score
import numpy as np

mse_train = np.sqrt(mean_squared_error(y_train,y_pred_train))
mse_test = np.sqrt(mean_squared_error(y_test,y_pred_test))

r2_train = r2_score(y_train,y_pred_train)
r2_test = r2_score(y_test,y_pred_test)

print(mse_train)
print(mse_test)
print(r2_train)
print(r2_test)

11167.763059779658
28438.109495224682
0.9790899692653484
0.8945643184113071


In [70]:
from sklearn.model_selection import GridSearchCV

param_grid = {'model__n_estimators':[100,200],'model__max_depth':[10,20,None]}

search = GridSearchCV(regression_pipeline,param_grid,cv=5)

search.fit(X_train,y_train)

print(search.best_params_)


{'model__max_depth': 20, 'model__n_estimators': 200}
